# Debug Notebook: Compact Ablation (Quick Run)

This notebook isolates dominant factors for speculative decoding performance by varying:
- `k` (proposal length)
- output length (`max_new_tokens`)
- prompt length bucket (short / medium / long)

Design goal: fast, GPU-only debugging run with compact but informative outputs.

In [ ]:
import gc
import os
import sys
from pathlib import Path

import pandas as pd
try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "torch is required for this notebook. Select a kernel/environment with PyTorch installed."
    ) from exc

# Resolve project root robustly.
search_roots = []
if '__file__' in globals():
    search_roots.append(Path(__file__).resolve().parent)
search_roots.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_roots:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src').exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError('Could not find project root containing src/.')

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Offline-first behavior.
os.environ['SPECDEC_HF_OFFLINE_FIRST'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# GPU-only enforcement.
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError('CUDA GPU is required for this notebook.')

from config import TARGET_MODEL_ID, DRAFT_MODELS, REGIMES
from baseline import run_baseline_sample
from speculative import load_model_on_device, speculative_decode_sample

results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print('GPU-only mode enabled')
print(f'CUDA device count: {torch.cuda.device_count()}')

In [ ]:
# Ablation controls (compact quick-run).
DEVICE = 'cuda:0'
TARGET_QUANT = 'fp16'
DRAFT_QUANT = 'fp16'
DRAFT_LABEL = '0.5B'
MODES = ['Deterministic']  # add 'Stochastic' if needed

K_VALUES = [2, 4, 8]
OUTPUT_LENGTHS = [12, 32, 64]
PROMPT_BUCKETS = ['short', 'medium', 'long']

BASE_PROMPTS = [
    'Why are leaves green? Answer in one sentence.',
    'What is an API? One-sentence answer.',
    'Define overfitting in machine learning in one sentence.',
    'Give one concise writing tip.',
]

N_PROMPTS_PER_BUCKET = 2

RAW_CSV = results_dir / 'ablation_compact_raw.csv'
SUMMARY_CSV = results_dir / 'ablation_compact_summary.csv'
DOMINANCE_CSV = results_dir / 'ablation_factor_dominance.csv'

print('Ablation settings:')
print(f'  device={DEVICE}, target_quant={TARGET_QUANT}, draft_quant={DRAFT_QUANT}, draft={DRAFT_LABEL}')
print(f'  modes={MODES}, k={K_VALUES}, output_lengths={OUTPUT_LENGTHS}, buckets={PROMPT_BUCKETS}')

In [ ]:
# Prompt-bucket constructor (vary prompt length while preserving task intent).
MEDIUM_PREFIX = (
    'Context: In practical ML systems, response quality, latency, and resource usage must be balanced. '
    'Provide a concise, technically accurate answer. '
)
LONG_PREFIX = ' '.join([
    'Context: System constraints include memory bandwidth, kernel launch overhead, and cache behavior.'
    'Prompt engineering can alter prefill cost and token distribution.'
    'Focus on clarity and avoid unnecessary verbosity.'
] * 8)

def make_prompt(base: str, bucket: str) -> str:
    if bucket == 'short':
        return base
    if bucket == 'medium':
        return f'{MEDIUM_PREFIX}\nQuestion: {base}'
    if bucket == 'long':
        return f'{LONG_PREFIX}\nQuestion: {base}'
    raise ValueError(f'Unknown bucket: {bucket}')

bucket_prompts = {}
for b in PROMPT_BUCKETS:
    bucket_prompts[b] = [make_prompt(p, b) for p in BASE_PROMPTS[:N_PROMPTS_PER_BUCKET]]

for b, ps in bucket_prompts.items():
    print(f'{b}: {len(ps)} prompts')

In [ ]:
# Model caches and single-run helper.
target_cache = {}
draft_cache = {}

def _get_target(quant: str):
    key = (TARGET_MODEL_ID, quant.lower(), DEVICE)
    if key not in target_cache:
        target_cache[key] = load_model_on_device(TARGET_MODEL_ID, device=DEVICE, quant_mode=quant.lower())
    return target_cache[key]

def _get_draft(quant: str):
    model_id = DRAFT_MODELS[DRAFT_LABEL]
    key = (model_id, quant.lower(), DEVICE)
    if key not in draft_cache:
        draft_cache[key] = load_model_on_device(model_id, device=DEVICE, quant_mode=quant.lower())
    return draft_cache[key]

def run_one(prompt: str, mode: str, k: int, max_new_tokens: int) -> dict:
    regime = REGIMES[mode.lower()]

    target_model, target_tok = _get_target(TARGET_QUANT)
    draft_model, _ = _get_draft(DRAFT_QUANT)

    # Prompt token length (prefill proxy).
    prompt_tokens = int(target_tok(prompt, return_tensors='pt', truncation=True, max_length=2048)['input_ids'].shape[1])

    base = run_baseline_sample(
        target_model,
        target_tok,
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=regime.temperature,
        top_p=regime.top_p,
    )

    spec = speculative_decode_sample(
        target_model,
        draft_model,
        target_tok,
        prompt,
        max_new_tokens=max_new_tokens,
        k=k,
        temperature=regime.temperature,
        top_p=regime.top_p,
        return_timing_breakdown=True,
    )

    base_s = float(base.get('latency_s', 0.0))
    spec_s = float(spec.get('latency_s', 0.0))
    draft_s = float(spec.get('draft_elapsed_s', 0.0))
    verify_s = float(spec.get('verify_elapsed_s', 0.0))
    other_s = max(spec_s - draft_s - verify_s, 0.0)

    proposed = float(spec.get('total_proposed', 0.0))
    accepted = float(spec.get('total_accepted', 0.0))
    alpha = (accepted / proposed) if proposed > 0 else 0.0
    speedup = (base_s / spec_s) if spec_s > 0 else 0.0

    return {
        'prompt_tokens': prompt_tokens,
        'baseline_latency_s': base_s,
        'spec_latency_s': spec_s,
        'speedup': speedup,
        'alpha': alpha,
        'draft_s': draft_s,
        'verify_s': verify_s,
        'other_s': other_s,
        'draft_share': (draft_s / spec_s) if spec_s > 0 else 0.0,
        'verify_share': (verify_s / spec_s) if spec_s > 0 else 0.0,
        'other_share': (other_s / spec_s) if spec_s > 0 else 0.0,
    }

In [ ]:
# Execute compact ablation grid.
rows = []
total_runs = len(MODES) * len(K_VALUES) * len(OUTPUT_LENGTHS) * len(PROMPT_BUCKETS) * N_PROMPTS_PER_BUCKET
done = 0

for mode in MODES:
    for k in K_VALUES:
        for out_len in OUTPUT_LENGTHS:
            for bucket in PROMPT_BUCKETS:
                prompts = bucket_prompts[bucket]
                for i, prompt in enumerate(prompts, start=1):
                    done += 1
                    print(f'[{done}/{total_runs}] mode={mode} k={k} out={out_len} bucket={bucket} sample={i}')
                    m = run_one(prompt=prompt, mode=mode, k=k, max_new_tokens=out_len)
                    rows.append({
                        'mode': mode,
                        'k': k,
                        'max_new_tokens': out_len,
                        'prompt_bucket': bucket,
                        'sample_idx': i,
                        **m,
                    })

df_raw = pd.DataFrame(rows)
df_raw.to_csv(RAW_CSV, index=False)
print(f'Saved raw runs -> {RAW_CSV}')
display(df_raw.head())

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Compact ablation summary table.
group_cols = ['mode', 'k', 'max_new_tokens', 'prompt_bucket']
summary = (
    df_raw.groupby(group_cols, as_index=False)
    .agg(
        n=('sample_idx', 'count'),
        prompt_tokens_mean=('prompt_tokens', 'mean'),
        speedup_mean=('speedup', 'mean'),
        alpha_mean=('alpha', 'mean'),
        spec_latency_mean_s=('spec_latency_s', 'mean'),
        baseline_latency_mean_s=('baseline_latency_s', 'mean'),
        draft_share_mean=('draft_share', 'mean'),
        verify_share_mean=('verify_share', 'mean'),
        other_share_mean=('other_share', 'mean'),
    )
)

for c in ['draft_share_mean', 'verify_share_mean', 'other_share_mean']:
    summary[c] = summary[c] * 100.0

summary.to_csv(SUMMARY_CSV, index=False)
print(f'Saved summary -> {SUMMARY_CSV}')
display(summary.sort_values(['mode', 'k', 'max_new_tokens', 'prompt_bucket']))

In [ ]:
# Dominance scores: how much variance each factor explains (main effects only).
def dominance_score(df: pd.DataFrame, y: str, factor: str) -> float:
    total_var = float(df[y].var(ddof=0))
    if total_var <= 0:
        return 0.0
    between = float(df.groupby(factor)[y].mean().var(ddof=0))
    return between / total_var

factors = ['k', 'max_new_tokens', 'prompt_bucket']
targets = ['speedup', 'spec_latency_s', 'alpha']
dom_rows = []
for y in targets:
    for f in factors:
        dom_rows.append({
            'metric': y,
            'factor': f,
            'dominance_score': dominance_score(df_raw, y, f),
        })

df_dom = pd.DataFrame(dom_rows)
df_dom.to_csv(DOMINANCE_CSV, index=False)
print(f'Saved dominance -> {DOMINANCE_CSV}')
display(df_dom.sort_values(['metric', 'dominance_score'], ascending=[True, False]))

print('\nHigher dominance_score means stronger main-effect contribution to variance.')

## Notes

- This notebook is for compact diagnosis, not final benchmark claims.
- To run faster: reduce `N_PROMPTS_PER_BUCKET` or fewer `OUTPUT_LENGTHS`.
- To compare regimes, set `MODES = ['Deterministic', 'Stochastic']`.
- You can reuse outputs in slides directly from:
  - `results/ablation_compact_summary.csv`
  - `results/ablation_factor_dominance.csv`